In [ ]:
淘天集团|https://talent.taotian.com/off-campus/position-list?lang=zh
淘宝闪购|https://talent.ele.me/off-campus/position-list?lang=zh&positionType=103
飞猪|https://career.fliggy.com/off-campus/position-list?lang=zh
阿里国际|https://aidc-jobs.alibaba.com/off-campus/position-list?lang=zh
阿里云|https://careers.aliyun.com/off-campus/position-list?lang=zh
通义实验室|https://careers-tongyi.alibaba.com/off-campus/position-list?lang=zh
钉钉|https://talent.dingtalk.com/off-campus/position-list?lang=zh
千问C端事业群|https://talent.quark.cn/off-campus/position-list?lang=zh
高德地图|https://talent.amap.com/off-campus/position-list?lang=zh
菜鸟集团|https://talent.cainiao.com/social-recruitment
虎鲸文娱集团|https://jobs.hujing-dme.com/off-campus/position-list?lang=zh
阿里健康|https://careers.alihealth.cn/off-campus/position-list?lang=zh
灵犀互娱|https://talent.lingxigames.com/off-campus/position-list?lang=zh
菜鸟驿站|https://talent-post.alibaba.com/off-campus/position-list?lang=zh
B站|https://jobs.bilibili.com/social/positions?code=03&type=3&onlyHotRecruit=1&page=1
米哈游|https://jobs.mihoyo.com/#/position?jobName=&competencyTypes%5B0%5D=5&competencyTypes%5B1%5D=8


# 测试

In [12]:
import requests
import json
from typing import List, Dict
# 新增：时间处理模块，用于筛选近两天数据
from datetime import datetime, timedelta

def get_163_jobs() -> List[Dict]:
    """
    爬取网易招聘岗位信息，筛选近两天更新的岗位，直接打印结果
    :return: 包含近两天岗位信息的列表
    """
    job_list = []
    # 完整请求头
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://hr.163.com/job-list.html",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "Content-Type": "application/json;charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": "https://hr.163.com"
    }
    
    api_url = "https://hr.163.com/api/hr163/position/queryPage"
    # 仅请求第1页，pageSize=200覆盖全部数据
    post_data = {
        "currentPage": 1,
        "pageSize": 200,
        "postType": "08",
        "workType": "0",
        "cityIdList":[229, 2, 138],
        "lang": "zh"
    }

    try:
        print("正在获取网易招聘岗位数据...")
        # 发送单次POST请求
        response = requests.post(
            api_url,
            headers=headers,
            json=post_data,
            timeout=15
        )
        response.raise_for_status()
        response_json = response.json()

        # 检查请求是否成功
        if response_json.get("code") != 200:
            print(f"请求失败: {response_json.get('msg')}")
            return job_list

        data = response_json.get("data")
        if not data:
            print("无数据返回")
            return job_list

        jobs = data.get("list", [])
        if not jobs:
            print("未获取到岗位信息")
            return job_list

        # 解析所有岗位信息（新增：更新时间戳，用于筛选）
        for job in jobs:
            job_id = job.get("id", "")
            detail_url = f"https://hr.163.com/job-detail.html?id={job_id}&lang=zh" if job_id else ""
            # 新增：获取岗位更新时间戳（毫秒级）
            update_time_stamp = job.get("updateTime", 0)
            
            job_info = {
                "岗位名称": job.get("name", ""),
                "岗位地址": ",".join(job.get("workPlaceNameList", [])),
                "职位描述": job.get("description", "").replace("\n", " "),
                "职位要求": job.get("requirement", "").replace("\n", " "),
                "岗位详情页URL": detail_url,
                "更新时间戳": update_time_stamp  # 用于时间筛选
            }
            job_list.append(job_info)

        # ===================== 核心修改：筛选近两天更新的岗位 =====================
        now = datetime.now()
        two_days_ago = now - timedelta(days=2)  # 计算48小时前的时间
        filtered_jobs = []
        
        for job in job_list:
            stamp = job["更新时间戳"]
            if not stamp:
                continue
            # 毫秒级时间戳转换为标准时间
            update_time = datetime.fromtimestamp(stamp / 1000)
            # 筛选：更新时间 >= 两天前
            if update_time >= two_days_ago:
                # 格式化时间，方便查看
                job["更新时间"] = update_time.strftime("%Y-%m-%d %H:%M:%S")
                filtered_jobs.append(job)
        # ======================================================================

        print(f"\n数据筛选完成！总数据：{len(job_list)} 条，近两天更新：{len(filtered_jobs)} 条\n")
        return filtered_jobs

    except requests.exceptions.RequestException as e:
        print(f"请求异常: {e}")
    except Exception as e:
        print(f"解析异常: {e}")

    return []

# 新增：格式化打印岗位信息
def print_jobs(jobs: List[Dict]):
    if not jobs:
        print("❌ 暂无近两天更新的岗位信息！")
        return
    
    # 遍历打印每个岗位，分隔线区分，清晰易读
    for index, job in enumerate(jobs, 1):
        print("-" * 80)
        print(f"【岗位 {index}】")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"岗位地址：{job['岗位地址']}")
        print(f"更新时间：{job['更新时间']}")
        print(f"职位描述：{job['职位描述']}")
        print(f"职位要求：{job['职位要求']}")
        print(f"详情链接：{job['岗位详情页URL']}")
    print("-" * 80)

if __name__ == "__main__":
    # 1. 获取并筛选近两天的岗位数据
    filtered_job_data = get_163_jobs()
    # 2. 直接打印结果
    print_jobs(filtered_job_data)

d:\Anaconda3\lib\site-packages\chardet\langbulgarianmodel.py:3588: RuntimeWarning: coroutine 'get_filtered_wangyi_jobs' was never awaited
  27: {  # 'ш'
d:\Anaconda3\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (4.0.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn(


正在获取网易招聘岗位数据...

数据筛选完成！总数据：119 条，近两天更新：3 条

--------------------------------------------------------------------------------
【岗位 1】
岗位名称：【平台】海外产品运营
岗位地址：广州市
更新时间：2026-03-30 15:20:01
职位描述：1、产品策略制定与执行： 负责产品的全生命周期运营，制定用户增长、留存及付费转化策略；基于用户反馈及数据分析，持续推动产品功能优化，提升用户体验与品牌口碑。 2、用户增长与生态建设：主动探索多样化的获客手段，驱动内外部渠道协同实现用户规模增长；负责全球用户社群运营，策划高质量社群活动，维护产品评分与玩家生态。 3、市场合作与活动运营：拓展行业资源，与游戏厂商等合作伙伴建立联系，推动联合营销、活动联动或功能嵌入等深度合作项目，提升产品市场渗透率。 4、机会洞察与策略预判： 保持对全球手游市场的高度敏锐，前瞻性捕捉爆款游戏上线、网络波动等市场机会点，快速输出差异化运营方案及本地化策略。 5、数据驱动与商业化探索： 建立并完善数据监控体系，通过深度分析用户行为数据驱动决策，挖掘有效的商业化增长点，持续提升运营效能。
职位要求：1、3年以上互联网工具类 App 或游戏运营经验，英语能够作为日常工作语言，具备成功的增长案例或海外项目运营背景，有跨文化协作经验者优先。 2、具备极强的探索意识和自驱动力，能快速捕捉并跟进全球爆款游戏上线等市场机会。 3、熟悉游戏玩家需求及痛点，对游戏工具类产品及相关技术原理有一定了解者优先。 4、具备扎实的数据分析能力，能够熟练运用相关工具辅助业务复盘与决策优化。 5、具备优秀的资源整合与沟通谈判能力，能独立、主动地推动跨团队及跨公司合作。 6、目标导向，抗压性强，能够适应快节奏的工作环境并高效达成项目目标。
详情链接：https://hr.163.com/job-detail.html?id=74769&lang=zh
--------------------------------------------------------------------------------
【岗位 2】
岗位名称：高级用户运营（阴阳师）
岗位地址：广州市
更新时间：2026-03-30 

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import date, timedelta
import time

def get_filtered_dji_jobs():
    # ===================== 核心配置 =====================
    # 大疆仅爬第一页
    BASE_URL = "https://we.dji.com/zh-CN/social?from=home_page&category=301_302&location=3100_4403&pageSize=100&page=1"
    # 排除关键词：硕士+工作年限
    EXCLUDE_KEYWORDS = ['硕士','3年','4年','5年','6年','7年','8年','9年','10年','三年','四年','五年']
    
    # ===================== 【你的代码风格】ARM Chromium 配置 =====================
    options = Options()
    # 必选参数（Docker+ARM 必备）
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    # 【关键】连接本地 Docker 中的 seleniarm 浏览器（容器间通信地址）
    driver = webdriver.Remote(
        command_executor="http://192.168.2.53:4444/wd/hub",
        options=options
    )
    driver.set_page_load_timeout(60)
    all_jobs = []

    # 获取近2天日期
    def get_valid_dates():
        today = date.today()
        yesterday = today - timedelta(days=1)
        return [today.strftime("%Y-%m-%d"), yesterday.strftime("%Y-%m-%d")]

    # 爬取岗位详情
    def crawl_job_detail(job_item):
        try:
            job_name = job_item.find_element(By.CLASS_NAME, "PositionCard_text__2BdZa").text.strip()
            keyword_text = job_item.find_element(By.CLASS_NAME, "PositionCard_keyword__FFaH5").text.strip()
            keyword_parts = [part.strip() for part in keyword_text.split("|")]
            city = keyword_parts[0]
            update_time = keyword_parts[-1]  # 直接获取日期：2026-03-18
            detail_url = job_item.find_element(By.TAG_NAME, "a").get_attribute("href")

            # 打开详情页
            main_handle = driver.current_window_handle
            driver.execute_script("window.open(arguments[0]);", detail_url)
            time.sleep(2)
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(2)

            # 提取任职要求
            requirement = ""
            subtitles = driver.find_elements(By.CLASS_NAME, "detail_subtitle__gOlwP")
            contents = driver.find_elements(By.CLASS_NAME, "detail_phases__PyEga")
            for i, sub in enumerate(subtitles):
                if "任职要求" in sub.text and i < len(contents):
                    requirement = contents[i].text.strip()

            driver.close()
            driver.switch_to.window(main_handle)
            return {
                "岗位名": job_name,
                "工作地点": city,
                "详情链接": detail_url,
                "更新时间": update_time,
                "岗位要求": requirement
            }
        except Exception:
            if len(driver.window_handles) > 1:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            return None

    # ===================== 主爬取逻辑 =====================
    try:
        valid_dates = get_valid_dates()
        driver.get(BASE_URL)
        time.sleep(5)
        
        # 获取岗位列表
        job_items = driver.find_elements(By.CLASS_NAME, "social_position_card__epffd")
        
        for item in job_items:
            job = crawl_job_detail(item)
            if job:
                all_jobs.append(job)

        # 双重筛选
        recent_jobs = [j for j in all_jobs if j["更新时间"] in valid_dates]
        final_jobs = [j for j in recent_jobs if not any(k in j["岗位要求"] for k in EXCLUDE_KEYWORDS)]
        
        return final_jobs
    finally:
        driver.quit()

# 运行并输出结果
if __name__ == "__main__":
    result = get_filtered_dji_jobs()
    print(f"\n✅ 筛选完成，符合条件岗位：{len(result)}")
    for job in result:
        print(job)


✅ 筛选完成，符合条件岗位：3
{'岗位名': '中/高级区域销售岗（大疆行业-欧洲/东南亚/中东非/拉美）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1782728415362752512', '更新时间': '2026-03-18', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语/韩语/德语/葡语/西语/阿拉伯语），接受海外长期外派；\n2. 具备销售、渠道管理等相关经验，熟悉平台商运作模式，有海外工作经验、无人机相关工作经验优先；\n3. 具备较强的市场分析和拓展能力、渠道管理能力、执行力、沟通协调能力；\n4. 工作认真负责、善沟通协调、心态开放，具备良好的应变能力和承压能力。'}
{'岗位名': '中/高级区域销售岗（消费级产品-欧洲/北美/东南亚/澳洲/巴西/墨西哥/印度/日本/韩国）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1755151167823581184', '更新时间': '2026-01-29', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语、韩语、西语、葡语等），接受海外长期外派；\n2. 熟悉消费电子海外渠道业务模式及操作方法；\n3. 具备较强的市场分析和拓展能力、渠道管理能力；\n4. 工作认真负责、善沟通协调，具备良好的应变能力和承压能力，心态开放。'}
{'岗位名': '中/高级区域销售岗（大疆农业-欧洲/北美/日韩/拉美/巴西/东南亚/中东非）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1870066161332559872', '更新时间': '2025-08-11', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语/韩语/德语/葡语/西语/阿拉伯语），接受海外长期外派；\n2. 具备销售、渠道管理等相关经验，熟悉平